In [62]:
import pandas as pd
import xml.etree.ElementTree as ET
import os, sys
import difflib
import ctypes
import stat
from pylibsrcml import srcml
from zss import simple_distance, Node
import zss

import subprocess

def get_xml_tree(code, dir='/home/egk204/PycharmProjects/code-clone-multilingual/storage'):
    def forc(j_path, c):
        f = open(j_path, "w")
        f.write(c)
        f.close()
    j_path = dir+'/temp/temp_td.java'
    xml_path = dir+'/temp/temp_td.java.xml'

    try:
        forc(j_path, code)
        r = subprocess.run(['srcml', j_path,'-o',xml_path], timeout=5)      
    except subprocess.TimeoutExpired as e:
        forc(j_path,'')
        r = subprocess.run(['srcml', j_path,'-o',xml_path], timeout=5) 
    tree = ET.parse(xml_path)
    rt = tree.getroot()      
    return rt


def rem(s):
    return str(s.tag).replace("{http://www.srcML.org/srcML/src}","")  

def get_avg_root_to_leaf_dist(current_node, tree_paths=[], val=0):           
    #process current node
    token = rem(current_node)
    val+=1

    if token != 'unit' :
        #special case, for current node, attribute, literal value should be added as children to make each path distinct
        cattr = current_node.attrib
        ctoken = current_node.text  
        if bool(cattr):
            for _, token_ in cattr.items():
                val+=1#"-"+token_                
        if bool(ctoken):
            val+=1#"-"+ctoken
    #visit all children            
    for child in current_node:   
        get_avg_root_to_leaf_dist(child, tree_paths, val)
    tree_paths.append(val)
    
    #root_to_leaf_dist = [len(i.split('-')) for i in tree_paths]
    return sum(tree_paths)/len(tree_paths)

rt = get_xml_tree('int i = 10')
rt

<Element '{http://www.srcML.org/srcML/src}unit' at 0x7f6a417339c0>

In [63]:
get_avg_root_to_leaf_dist(rt, [])

4.0

In [64]:
o = pd.read_pickle('/home/egk204/PycharmProjects/code-clone-multilingual/storage/data_original.pkl')
ja = o[o.language_extension=='java']
py = o[o.language_extension=='py']

In [96]:
px = []
for i in py.code:
    px.extend(i.split('\n'))

import ast
def check_py_ast(c):
    try:
        ast.parse(c)
        return True
    except:
        return False
    
pxp = [s for s in px if check_py_ast(s) ]

In [108]:
jx = []
for i in ja.code:
    jx.extend(i.split(';'))

import javalang
def check_ja_ast(c):
    try:
        tokens = javalang.tokenizer.tokenize(c+";")
        parser = javalang.parser.Parser(tokens)
        parser.parse_expression()
        return True
    except:
        return False
    
jxp = [s for s in jx if check_ja_ast(s) ]

In [109]:
len(jx), len(jxp)


(1168294, 382668)